# Laboratorio: sentimiento de reseñas en español

Este cuaderno compara dos modelos ya entrenados sobre las mismas frases y luego ejecuta una misma predicción localmente con `pipeline` y de forma remota con la API Gradio de Hugging Face mediante `gradio_client`. La Space usa Gradio nativo porque ZeroGPU requiere una función Gradio decorada con `@spaces.GPU`; por eso el cliente remoto de esta Space no es `InferenceClient`. No se entrena ningún modelo. Ejecutá las celdas en orden y guardá los resultados reales de tu equipo; los tiempos dependen de la conexión, el hardware, la caché y la disponibilidad de ZeroGPU.


## Modelos candidatos y fuentes

- [`cardiffnlp/twitter-xlm-roberta-base-sentiment`](https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment): clasifica sentimiento en negativo, neutro o positivo. Su model card describe XLM-RoBERTa entrenado con unos 198 millones de tweets y ajuste de sentimiento en ocho idiomas, incluido español. El Hub informa un archivo de pesos de 1.112.271.561 bytes (aprox. 1,04 GiB). La tarjeta no especifica todos los corpus de ajuste.
- [`pysentimiento/robertuito-sentiment-analysis`](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis): clasifica POS/NEU/NEG. Su tarjeta informa RoBERTuito preentrenado con más de 500 millones de tweets en español y ajuste con TASS 2020 (aprox. 5.000 tweets de distintas variedades del español). El archivo `model.safetensors` del Hub ocupa 435.189.428 bytes (aprox. 415 MiB). La tarjeta limita el uso a investigación científica y uso no comercial.

Los tamaños son los archivos de pesos indicados por el Hub al preparar este material; comprobá los model cards antes de entregar porque pueden cambiar. Para evitar depender de créditos de Inference Providers, el bloque remoto de este cuaderno usa una Space pública propia con ZeroGPU y la API Gradio de `gradio_client`.


In [ ]:
import os
import time

import pandas as pd
from gradio_client import Client
from transformers import pipeline

MODELS = {
    "Cardiff XLM-R": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    "RoBERTuito": "pysentimiento/robertuito-sentiment-analysis",
}
SELECTED_MODEL = MODELS["Cardiff XLM-R"]
print("Modelos candidatos:", MODELS)


## Conjunto pequeño de prueba

Las etiquetas esperadas son una anotación manual para este ejercicio, no una verdad universal. Hay diez ejemplos (más del mínimo de ocho), incluyendo ironía, una reseña en inglés, texto muy corto y jerga rioplatense. Revisá si las etiquetas esperadas te parecen razonables y explicá las limitaciones de un conjunto tan pequeño.


In [ ]:
EJEMPLOS = [
    {"frase": "El pedido llegó rápido y todo estaba riquísimo. Volvería a comprar.", "esperada": "Positivo"},
    {"frase": "Me atendieron con mucha paciencia; excelente servicio.", "esperada": "Positivo"},
    {"frase": "La comida estaba bien, sin nada especial.", "esperada": "Neutro"},
    {"frase": "El paquete llegó ayer.", "esperada": "Neutro"},
    {"frase": "Esperé una hora y me entregaron el pedido frío.", "esperada": "Negativo"},
    {"frase": "No me respondieron el reclamo y nadie se hizo cargo.", "esperada": "Negativo"},
    {"frase": "Genial, otra vez me cancelaron el pedido a último minuto. Un lujo.", "esperada": "Negativo"},
    {"frase": "The delivery was late and the food arrived cold.", "esperada": "Negativo"},
    {"frase": "Malísimo.", "esperada": "Negativo"},
    {"frase": "Che, re piola el lugar, alta atención.", "esperada": "Positivo"},
]

ALIAS_ETIQUETAS = {
    "positive": "Positivo", "pos": "Positivo", "positivo": "Positivo",
    "neutral": "Neutro", "neu": "Neutro", "neutro": "Neutro",
    "negative": "Negativo", "neg": "Negativo", "negativo": "Negativo",
}

def normalizar_etiqueta(etiqueta):
    texto = str(etiqueta).strip()
    return ALIAS_ETIQUETAS.get(texto.lower(), texto)

pd.DataFrame(EJEMPLOS)


## Comparación local con `pipeline`

Cada modelo se descarga/carga una vez y recibe exactamente las mismas diez frases. Se registra por separado el tiempo de carga y el tiempo total de inferencia del conjunto. En una primera ejecución, la descarga puede dominar la carga; en las siguientes, los archivos pueden estar en caché. Si tu computadora no tiene memoria suficiente, ejecutá el cuaderno en Colab.


In [ ]:
PIPELINES = {}
tiempos_carga = {}
filas_resultados = []

for nombre_modelo, model_id in MODELS.items():
    inicio_carga = time.perf_counter()
    clasificador = pipeline(
        "text-classification",
        model=model_id,
        device=-1,
        use_fast=False,
    )
    tiempos_carga[nombre_modelo] = time.perf_counter() - inicio_carga
    PIPELINES[model_id] = clasificador

    inicio_inferencia = time.perf_counter()
    for ejemplo in EJEMPLOS:
        predicciones = clasificador(ejemplo["frase"])
        mejor = max(predicciones, key=lambda item: item["score"])
        predicha = normalizar_etiqueta(mejor["label"])
        filas_resultados.append({
            "modelo": nombre_modelo,
            "frase": ejemplo["frase"],
            "etiqueta esperada": ejemplo["esperada"],
            "etiqueta predicha": predicha,
            "puntaje": float(mejor["score"]),
            "¿acertó?": predicha == ejemplo["esperada"],
        })
    tiempos_carga[f"{nombre_modelo} · inferencia de 10 frases"] = (
        time.perf_counter() - inicio_inferencia
    )

resultados = pd.DataFrame(filas_resultados)
display(resultados)
display(pd.Series(tiempos_carga, name="segundos medidos").to_frame())


## Lectura de la comparación

La columna `puntaje` expresa el score que devuelve cada modelo para su etiqueta principal; no es una garantía ni una probabilidad necesariamente calibrada. Observá especialmente las frases difíciles. Un error puede deberse a la ironía, al contexto ausente, a una frase muy corta, a jerga local o a diferencias entre las reseñas de este ejercicio y los tweets usados para ajustar los modelos.

No elijas un modelo solo por el porcentaje de aciertos de diez frases: el conjunto es pequeño y fue anotado manualmente. Considerá también idioma y dominio, tamaño de los pesos, licencia, proveedor disponible y límites de uso. Anotá aquí los casos concretos que revisarás en el informe; no inventes resultados.


In [ ]:
resumen = resultados.groupby("modelo")["¿acertó?"].agg(["sum", "count", "mean"])
resumen = resumen.rename(columns={"sum": "aciertos", "count": "casos", "mean": "proporción de aciertos"})
display(resumen)


## La misma frase con `pipeline` y la API Gradio remota

`pipeline` ejecuta el modelo en este entorno local: la primera carga puede descargar más de 1 GiB de pesos de Cardiff, ocupar memoria y tardar más que las inferencias posteriores. `gradio_client` envía el texto a la API Gradio de tu Space y recibe las predicciones; no descarga los pesos en esta computadora. El tiempo remoto incluye red, cola y ejecución en ZeroGPU, así que no es una comparación de hardware equivalente.

Para la llamada remota, definí `INFERENCE_ENDPOINT_URL` con la URL pública de tu Space, por ejemplo `https://USUARIO-NOMBRE-SPACE.hf.space/`, y `HF_TOKEN` con un token de lectura de Hugging Face. El token autentica el acceso; no se comparte con el modelo como texto. La celda falla con un mensaje explícito si falta una variable. ZeroGPU puede tener espera o cuota limitada.


In [ ]:
FRASE_BENCHMARK = "No me respondieron el reclamo y nadie se hizo cargo."

# El pipeline de Cardiff ya se cargó en la comparación; medimos una llamada local caliente.
pipeline_cardiff = PIPELINES[SELECTED_MODEL]
inicio = time.perf_counter()
salida_pipeline = pipeline_cardiff(FRASE_BENCHMARK)
segundos_pipeline = time.perf_counter() - inicio
print("Tipo de retorno de pipeline:", type(salida_pipeline))
print("Tipo del primer elemento:", type(salida_pipeline[0]))
print("Etiqueta mediante diccionario:", salida_pipeline[0]["label"])
print("Puntaje mediante diccionario:", salida_pipeline[0]["score"])
print(f"Tiempo de inferencia local (modelo ya cargado): {segundos_pipeline:.4f} s")


In [ ]:
endpoint_url = os.getenv("INFERENCE_ENDPOINT_URL")
token = os.getenv("HF_TOKEN")
if not endpoint_url:
    raise RuntimeError(
        "Falta INFERENCE_ENDPOINT_URL. Configurá la URL pública de tu "
        "Space pública como variable de entorno antes de ejecutar esta celda."
    )
if not token:
    raise RuntimeError("Falta HF_TOKEN; configurá un token de lectura como secreto.")

cliente = Client(
    src=endpoint_url,
    hf_token=token,
    httpx_kwargs={"timeout": 120},
)
inicio = time.perf_counter()
salida_remota = cliente.predict(FRASE_BENCHMARK, api_name="/clasificar")
segundos_remoto = time.perf_counter() - inicio
print("Tipo de retorno de gradio_client:", type(salida_remota))
print("Tipo del primer elemento:", type(salida_remota[0]))

primer_resultado = salida_remota[0]
etiqueta_remota = primer_resultado["label"]
puntaje_remoto = primer_resultado["score"]

print("Etiqueta remota:", etiqueta_remota)
print("Puntaje remoto:", puntaje_remoto)
print(f"Tiempo de respuesta remoto (incluye red): {segundos_remoto:.4f} s")
display(pd.DataFrame([
    {"forma": "pipeline (modelo cargado)", "segundos": segundos_pipeline},
    {"forma": "gradio_client (llamada remota)", "segundos": segundos_remoto},
]))


## Qué llevar al informe

Después de ejecutar el cuaderno, guardá: (1) la tabla completa de frases y resultados para cada modelo; (2) los aciertos y casos de error que realmente observaste; (3) el tipo de retorno y la forma de acceder a etiqueta y puntaje; (4) los tiempos medidos, aclarando que el local es una inferencia caliente y el remoto incluye red; (5) cualquier error real, con mensaje completo, lugar, causa y pasos de solución. No presentes los tiempos de otra computadora como propios. Antes de entregar, quitá secretos de las salidas y comprobá que el notebook se pueda reiniciar y ejecutar de principio a fin.
